In [11]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath("../"))

from scripts.util.getters import get_dataset_path, get_prediction_path

def review_predictions(model_name: str = "t5-small", num_samples: int = 5):
    """
    Print the first `num_samples` input-output-prediction examples across all enhancements.

    For each sample:
    - Shows input
    - Shows expected output (from raw dataset)
    - Shows predictions for raw, srl, kga, rbm, srt
    """
    enhancements = ["raw", "srl", "kga", "rbm", "srt"]

    # --- Load input and expected output from raw dataset ---
    raw_path = get_dataset_path("raw", "test")
    # Load raw test data (assume at least input/output, possibly roles)
    raw_df = pd.read_csv(raw_path, sep="\t", header=None)

    # Assign column names based on actual number of columns
    if raw_df.shape[1] == 2:
        raw_df.columns = ["input", "expected"]
    elif raw_df.shape[1] == 3:
        raw_df.columns = ["input", "expected", "roles"]
    else:
        raise ValueError(f"Unexpected number of columns in raw dataset: {raw_df.shape[1]}")


    for i in range(num_samples):
        print(f"\n================== Sample {i + 1} ==================")
        print(f"Input:\n{raw_df.loc[i, 'input']}")
        print(f"Expected Output:\n{raw_df.loc[i, 'expected']}")
        print("------")

        for enh in enhancements:
            try:
                pred_path = get_prediction_path(model_name, "test", enh)
                with open(pred_path, "r", encoding="utf-8") as f:
                    predictions = f.readlines()
                pred = predictions[i].strip()
            except Exception as e:
                pred = f"[Error loading prediction: {e}]"

            print(f"{enh} Output:\n{pred}")


In [12]:
review_predictions("t5-small", num_samples=5)


================== Sample 1 ==================
Input:
Mila liked that the cake was offered to Emma .
Expected Output:
* cake ( x _ 4 ) ; like . agent ( x _ 1 , Mila ) AND like . ccomp ( x _ 1 , x _ 6 ) AND offer . theme ( x _ 6 , x _ 4 ) AND offer . recipient ( x _ 6 , Emma )
------
raw Output:
* book ( x _ 4 ) ; wire. agent ( x _ 1, Camila ) AND wire. theme ( x _ 1, x _ 3 ) AND wire. recipient ( x _ 1, Emma )
srl Output:
* book in_distribution. in_distribution. in_distribution.
kga Output:
* book ( x _ 4 ) ; wire. agent ( x _ 1, Camila ) AND wire. theme ( x _ 1, x _ 3 ) AND wire. recipient ( x _ 1, Emma )
rbm Output:
* book ( x _ 4 ) ; wire. agent ( x _ 1, Camila ) AND wire. theme ( x _ 1, x _ 3 ) AND wire. recipient ( x _ 1, Emma )
srt Output:
* book in_distribution. in_distribution. in_distribution.

================== Sample 2 ==================
Input:
A coach supported that the cake was snapped .
Expected Output:
* cake ( x _ 5 ) ; coach ( x _ 1 ) AND support . agent ( x _ 2 , x 